In [ ]:
"""
# initial fuzzy query from radiologist:

* Look at each tissue seperately
* if we have tissues with textures that are very different from the bulk of the tissue
* Then these people we should call out as anomalous

"""

"""
# Multi objective learning over intermediate representations
Images(img_id; img:(3,1024,1024))
# we have pathology labels over all training images
PathologyLabels(img_id; y_p)

# we have segmentation and anomalous texture labels over a small subset of the training images
SegmentationLabels(img_id, tissue_id; img:(3,1024,1024))
AnomalousTextureLabels(img_id, tissue_id; y_at)

# call a segment anything model (partially frozen) to get a segmentation for each tissue
StackedSegments(img_id; SegmentAnything(img):(|tissues|,3,1024,1024)) :- Images(img_id;img)

# we need to split the stacked segments into seperate entities, TBD decide on syntax to split and merge embeddings
Segmentation(img_id, [tissue_id:unsqueeze] ; img) :- StackedSegments(img_id,tissue_id;img)

# use an unfrozen image net backbone as a start to get learnable texture embeddings
# we compute the average texture over all tissues
AverageTexture(img_id ; avg(ImgNetHead(z)) ) :- Segmentation(img_id,tissue_id; z)
# and use an MLP to predict which tissues are anomalous compared to the average
AnomalousTexture(img_id,tissue_id ; MLP(out_dim=1)(concat(t1,t_avg))) :- Segmentation(img_id,tissue_id; t1), AverageTexture(img_id; t_avg)

# naive pathology prediction is just the max anomaly score over all tissues
Pathology(img_id ; max(y)) :- AnomalousTexture(img_id,tissue_id; y)

# we compute a loss over all label types, both e2e labels and intermediate labels (texture and segmentation)
SL(;sum(min(BoundaryLoss(s,s')))) :- Segmentation(img_id,tissue_id; s), SegmentationLabels(img_id,tissue_id2; s').
TL(;sum(CrossEntropy(y,y'))) :- AnomalousTexture(img_id,tissue_id; y), AnomalousTextureLabels(img_id,tissue_id; y').
PL(;sum(CrossEntropy(y,y'))) :- Pathology(img_id; y), PathologyLabels(img_id; y').

# We train on a combined loss, pushing the pathology network, to use intermediate concepts of texture and segmentation
# that map our small subset of intermediate labels 
?fit(; a_1* pathology_loss + a_2*texture_loss + a_3*segment_loss ):- 
    SL(;segmentation_loss),
    TL(;texture_loss),
    PL(;pathology_loss).


"""